In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%load_ext autoreload
%autoreload 2


from shared_modules.parser_functions import PlanParser, DocumentParser
from shared_modules.retriever import Retriever
from shared_modules.embeddings import get_embeddings
from services.procurement_reference_registry import ProcurementReferenceRegistry

In [22]:
import re
from pathlib import Path
from typing import Any, Dict, Optional, List

from shared_modules.parser_functions import (
    DocumentParser,
    PlanParser,
    parse_okpd_entries,
    parse_ktry_entries,
    _clean_keyword_dict,
    _extract_keyword_windows,
)
from shared_modules.retriever import Retriever, BM25TextRetriever

from latest_model.docs_parsing import (
    _parse_plan_points,
    _parse_contract_points,
    _parse_contract_characteristics,
    _parse_ooz_points,
    _parse_zapiska_text,
    _parse_onmck_text,
    _parse_onmck_pricies,
)
from latest_model.rag_processing import process_rag_points
from latest_model.smart_processing import process_smart_points
from latest_model.check_registry import get_regestry_response_okpd_ktry

pack_dir = project_root / "doci_primery" / "PACK_06_05"

contract_path = pack_dir / "4_Проект_контракта_шины_и_комплектующие.docx"
plan_path = pack_dir / "1_Заявка_на_включение_в_план_график.docx"
zapiska_path = pack_dir / "5. Пояснительная записка (1).docx"
OOZ_path = pack_dir / "3. ООЗ автошины и комплектующие.docx"
ONMCK_path = pack_dir / "2_ОЦК_метод_сопостовимых_рыночных_цен_без_учета_ПП_1875_.docx"
Obrasheniye_path = pack_dir / "0. Обращение ПД (1).docx"
# contract_path = Path(r"C:\Users\egorg\Documents\RAG_РјРёРЅС†РёС„СЂС‹\git_repo_clone\Documents-verification-Mintsifry\doci_primery\PACK_06_05\4_РџСЂРѕРµРєС‚_РєРѕРЅС‚СЂР°РєС‚Р°_С€РёРЅС‹_Рё_РєРѕРјРїР»РµРєС‚СѓСЋС‰РёРµ.docx")
# plan_path = Path(r"C:\Users\egorg\Documents\RAG_РјРёРЅС†РёС„СЂС‹\git_repo_clone\Documents-verification-Mintsifry\doci_primery\PACK_06_05\1_Р—Р°СЏРІРєР°_РЅР°_РІРєР»СЋС‡РµРЅРёРµ_РІ_РїР»Р°РЅ_РіСЂР°С„РёРє.docx")
# zapiska_path = Path(r"C:\\Users\\egorg\\Documents\\RAG_РјРёРЅС†РёС„СЂС‹\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\PACK_06_05\5. РџРѕСЏСЃРЅРёС‚РµР»СЊРЅР°СЏ Р·Р°РїРёСЃРєР° (1).docx")
# OOZ_path = Path(r"C:\\Users\\egorg\\Documents\\RAG_РјРёРЅС†РёС„СЂС‹\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\PACK_06_05\3. РћРћР— Р°РІС‚РѕС€РёРЅС‹ Рё РєРѕРјРїР»РµРєС‚СѓСЋС‰РёРµ.docx")
# ONMCK_path = Path(r"C:\\Users\\egorg\\Documents\\RAG_РјРёРЅС†РёС„СЂС‹\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\PACK_06_05\2_РћР¦Рљ_РјРµС‚РѕРґ_СЃРѕРїРѕСЃС‚РѕРІРёРјС‹С…_СЂС‹РЅРѕС‡РЅС‹С…_С†РµРЅ_Р±РµР·_СѓС‡РµС‚Р°_РџРџ_1875_.docx")
# Obrasheniye_path = Path(r"C:\\Users\\egorg\\Documents\\RAG_РјРёРЅС†РёС„СЂС‹\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\PACK_06_05\0. РћР±СЂР°С‰РµРЅРёРµ РџР” (1).docx")

In [32]:
table_ktry_names, table_characteristics, ktry_codes = _parse_contract_characteristics(OOZ_path)

In [33]:
parser_contract = DocumentParser(OOZ_path)
table_characteristics, ktry_codes = parser_contract.extract_tables_characteristics(
    keywords=["№", "КТРУ", "Наименование характеристики", "Значение характеристики"]
)
table_characteristics

{'№1. 22.11.11.000-00000007': {'Категория использования шины': 'Обычная (дорожная)',
  'Сезонность использования шины': 'Летняя',
  'Назначение': 'Легковая',
  'Номинальная ширина профиля': '185',
  'Номинальное отношение высоты профиля шины к ее ширине': '65',
  'Номинальный посадочный диаметр обода': '15',
  'Способ герметизации шины': 'Бескамерная',
  'Индекс категории скорости': 'H',
  'Индекс нагрузки': '92'},
 '№2. 22.11.11.000-00000004': {'Категория использования шины': 'Зимняя',
  'Назначение': 'Легковая',
  'Номинальная ширина профиля': '185',
  'Номинальное отношение высоты профиля шины к ее ширине': '65',
  'Номинальный посадочный диаметр обода': '15',
  'Способ герметизации шины': 'Бескамерная',
  'Индекс категории скорости': 'Т',
  'Индекс нагрузки': '92'},
 '№3. 22.11.11.000-00000007': {'Категория использования шины': 'Обычная (дорожная)',
  'Сезонность использования шины': 'Летняя',
  'Назначение': 'для внедорожника',
  'Номинальная ширина профиля': '225',
  'Номинальное

In [34]:
ktry_codes

{'№1. 22.11.11.000-00000007',
 '№10. 28.22.13.120',
 '№11. 27.20.21.000-00000021',
 '№12. 27.20.21.000-00000021',
 '№2. 22.11.11.000-00000004',
 '№3. 22.11.11.000-00000007',
 '№4. 29.32.30.220',
 '№5. 29.32.30.220',
 '№6. 29.32.30.220',
 '№7. 28.13.28.190',
 '№8. 28.24.11.000',
 '№9. 25.73.30.176'}

In [25]:
table_ktry_names

'| № п/п: 1 | ОКПД2/; КТРУ: 22.11.11.000-00000007 - Шина пневматическая для легкового автомобиля |\n| № п/п: 2 | ОКПД2/; КТРУ: 22.11.11.000-00000004 - Шина пневматическая для легкового автомобиля |\n| № п/п: 3 | ОКПД2/; КТРУ: 22.11.11.000-00000007 - Шина пневматическая для легкового автомобиля |\n| № п/п: 4 | ОКПД2/; КТРУ: 29.32.30.220 - Колеса, ступицы и их детали |\n| № п/п: 5 | ОКПД2/; КТРУ: 29.32.30.220 - Колеса, ступицы и их детали |\n| № п/п: 6 | ОКПД2/; КТРУ: 29.32.30.220 - Колеса, ступицы и их детали |\n| № п/п: 7 | ОКПД2/; КТРУ: 28.13.28.190 - Компрессоры прочие, не включенные в другие группировки |\n| № п/п: 8 | ОКПД2/; КТРУ: 28.24.11.000 - Инструменты ручные электрические |\n| № п/п: 9 | ОКПД2/; КТРУ: 25.73.30.176 - Головки сменные и принадлежности к ним в наборах и россыпью |\n| № п/п: 10 | ОКПД2/; КТРУ: 28.22.13.120 - Механизмы подъемные, используемые для подъема транспортных средств |\n| № п/п: 11 | ОКПД2/; КТРУ: 27.20.21.000-00000021 - Аккумулятор свинцовый для запуска п

In [24]:
# print(table_ktry_names)
table_characteristics

{'58.29.11.000-00000003': {'РљР»Р°СЃСЃ РїСЂРѕРіСЂР°РјРј РґР»СЏ СЌР»РµРєС‚СЂРѕРЅРЅС‹С… РІС‹С‡РёСЃР»РёС‚РµР»СЊРЅС‹С… РјР°С€РёРЅ Рё Р±Р°Р· РґР°РЅРЅС‹С…': '(12.10) РџСЂРѕРіСЂР°РјРјРЅРѕРµ РѕР±РµСЃРїРµС‡РµРЅРёРµ РґР»СЏ СЂРµС€РµРЅРёСЏ РѕС‚СЂР°СЃР»РµРІС‹С… Р·Р°РґР°С‡ РІ РѕР±Р»Р°СЃС‚Рё РёРЅС„РѕСЂРјР°С†РёРё Рё СЃРІСЏР·Рё',
  'РЎРїРѕСЃРѕР± РїСЂРµРґРѕСЃС‚Р°РІР»РµРЅРёСЏ': 'РљРѕРїРёСЏ СЌР»РµРєС‚СЂРѕРЅРЅРѕРіРѕ СЌРєР·РµРјРїР»СЏСЂР°',
  'Р’РёРґ Р»РёС†РµРЅР·РёРё': 'РџСЂРѕСЃС‚Р°СЏ (РЅРµРёСЃРєР»СЋС‡РёС‚РµР»СЊРЅР°СЏ)',
  'РњРѕРЅРёС‚РѕСЂРёРЅРі С‚РѕС‡РµРє Р±РµСЃРїСЂРѕРІРѕРґРЅРѕРіРѕ РґРѕСЃС‚СѓРїР°': 'РќР°Р»РёС‡РёРµ',
  'РћР±РЅРѕРІР»РµРЅРёРµ РјРёРєСЂРѕРїСЂРѕРіСЂР°РјРјРЅРѕРіРѕ РѕР±РµСЃРїРµС‡РµРЅРёСЏ С‚РѕС‡РµРє Р±РµСЃРїСЂРѕРІРѕРґРЅРѕРіРѕ РґРѕСЃС‚СѓРїР°': 'РќР°Р»РёС‡РёРµ',
  'РЈРїСЂР°РІР»РµРЅРёРµ Рё РЅР°СЃС‚СЂРѕР№РєР° С‚РѕС‡РµРє Р±РµСЃРїСЂРѕРІРѕРґРЅРѕРіРѕ РґРѕСЃС‚СѓРїР°': 'РќР°Р»РёС‡РёРµ',
  'РЎСЂРѕРє РґРµР№СЃС‚РІРёСЏ Р»РёС†РµРЅР·РёРё': 'Р‘РµСЃСЃСЂРѕС‡РЅРѕ',
  'РўРѕРІР°СЂРЅС‹Р№ Р·РЅР°Рє': 'ELTEX*'}}

In [14]:
%load_ext autoreload
%autoreload 2

from latest_model.check_registry import get_regestry_response_okpd_ktry, compare_characteristics
from services.procurement_reference_registry import ProcurementReferenceRegistry

def filter_plan_points(plan_points: List[str], keywords: List[str]) -> List[str]:
    plan_points_use = [
        plan_point
        for plan_point in plan_points
        if any(keyword.lower() in plan_point.lower() for keyword in keywords)
    ]
    return plan_points_use

REGISTRY_DIR = r"data\parsed_tables"
registry = ProcurementReferenceRegistry(REGISTRY_DIR)
# registry.get_ktru_common_info("26.30.23.000-00000016")
# registry.get_ktru_characteristics(ktry_codes[0])

plan_points = _parse_plan_points(plan_path)

procurement_method = filter_plan_points(
    plan_points,
    ["Способ выбора поставщика", "Способ выбора поставщика/исполнителя"],
)

okdp_plan = filter_plan_points(
    plan_points,
    ["ОКПД"],
)[0].split(":")[1].split("-")[0].strip()

characteristics_compare_result = compare_characteristics(
    OOZ_path,
    procurement_method,
    okdp_plan,
    REGISTRY_DIR,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
okdp_plan
procurement_method

['Способ выбора поставщика/исполнителя: Единственный поставщик (на бумаге)']

In [20]:
characteristics_compare_result

{}

In [6]:
import numpy as np
result = _parse_onmck_pricies(ONMCK_path)

print(result)

№1 Шина пневматическая для легкового автомобиля            | <ok>коэффициент вариации:  6.54%</ok> | Цены: [5462.0, 6226.68, 5844.34]
№2 Шина пневматическая для легкового автомобиля            | <ok>коэффициент вариации:  6.54%</ok> | Цены: [7650.0, 8721.0, 8185.5]
№3 Шина пневматическая для легкового автомобиля            | <ok>коэффициент вариации:  6.54%</ok> | Цены: [11818.0, 13472.52, 12645.26]
№4 Диск колесный                                           | <ok>коэффициент вариации:  6.54%</ok> | Цены: [11038.0, 12583.32, 11810.66]
№5 Диск колесный                                           | <ok>коэффициент вариации:  6.54%</ok> | Цены: [13648.0, 15558.72, 14603.36]
№6 Диск колесный                                           | <ok>коэффициент вариации:  6.54%</ok> | Цены: [5560.0, 6338.4, 5949.2]
№7 Компрессор автомобильный                                | <ok>коэффициент вариации:  6.54%</ok> | Цены: [7985.0, 9102.9, 8543.95]
№8 Гайковерт бесщеточный ударный                          

In [47]:
contract_points = _parse_contract_points(contract_path)
print(contract_points)

РќР°РёРјРµРЅРѕРІР°РЅРёРµ,; РћРљРџР”2/РљРўР РЈ
РџСЂРѕРіСЂР°РјРјРЅРѕРµ РѕР±РµСЃРїРµС‡РµРЅРёРµ,; РљРўР РЈ 58.29.11.000-00000003


In [78]:
ooz_points = _parse_ooz_points(OOZ_path)
print(ooz_points)

РќР°РёРјРµРЅРѕРІР°РЅРёРµ,; РћРљРџР”2/РљРўР РЈ
РџСЂРѕРіСЂР°РјРјРЅРѕРµ РѕР±РµСЃРїРµС‡РµРЅРёРµ,; РљРўР РЈ 58.29.11.000-00000003
| РќР°РёРјРµРЅРѕРІР°РЅРёРµ, ; РћРљРџР”2/РљРўР РЈ: РџСЂРѕРіСЂР°РјРјРЅРѕРµ РѕР±РµСЃРїРµС‡РµРЅРёРµ,; РљРўР РЈ 58.29.11.000-00000003 | РљРѕР»РёС‡РµСЃС‚РІРѕ, С€С‚СѓРє: 506 |


In [56]:
smart_keywords = [
    "РљРѕРґ РћРљРџР”",
    "РљРѕРґ РїРѕР·РёС†РёРё РљРўР РЈ",
    "РљРѕР»РёС‡РµСЃС‚РІРѕ",
]
plan_points = _parse_plan_points(plan_path)
plan_points_use = filter_plan_points(plan_points, smart_keywords)
parse_ktry_entries(plan_points_use[1])

[{'ktru_code': '58.29.11.000-00000003',
  'name': 'РЈ РґР°РЅРЅРѕРіРѕ РљРўР РЈ РЅРµ СѓРєР°Р·Р°РЅРѕ РёРјСЏ'}]

In [66]:
contract_points = _parse_contract_points(contract_path)
parser_contract = DocumentParser(contract_path)
parser_contract.extract_tables_columns(keywords=["РќР°РёРјРµРЅРѕРІР°РЅРёРµ,", "РљРѕР»РёС‡РµСЃС‚РІРѕ"])

'| РќР°РёРјРµРЅРѕРІР°РЅРёРµ, ; РћРљРџР”2/РљРўР РЈ: РџСЂРѕРіСЂР°РјРјРЅРѕРµ РѕР±РµСЃРїРµС‡РµРЅРёРµ,; РљРўР РЈ 58.29.11.000-00000003 | РљРѕР»РёС‡РµСЃС‚РІРѕ, С€С‚СѓРє: 506 |'

In [67]:
contract_points

'РќР°РёРјРµРЅРѕРІР°РЅРёРµ,; РћРљРџР”2/РљРўР РЈ\nРџСЂРѕРіСЂР°РјРјРЅРѕРµ РѕР±РµСЃРїРµС‡РµРЅРёРµ,; РљРўР РЈ 58.29.11.000-00000003\n| РќР°РёРјРµРЅРѕРІР°РЅРёРµ, ; РћРљРџР”2/РљРўР РЈ: РџСЂРѕРіСЂР°РјРјРЅРѕРµ РѕР±РµСЃРїРµС‡РµРЅРёРµ,; РљРўР РЈ 58.29.11.000-00000003 | РљРѕР»РёС‡РµСЃС‚РІРѕ, С€С‚СѓРє: 506 |'

In [36]:
from latest_model.ai_service import get_ai_service
from docx import Document

ai_service = get_ai_service()
result = ai_service.process_query(
    plan_path = plan_path,
    contract_path = contract_path,
    ooz_path = OOZ_path,
    zapiska_path = zapiska_path,
    ONMCK_path = ONMCK_path,
    Obrasheniye_path = Obrasheniye_path,
)

ai_response = result["ai_response"]

output_path = project_root / "latest_model" / "latest_analysis.docx"

doc = Document()
doc.add_paragraph(ai_response)
doc.save(output_path)

print("Saved to:", output_path)
print()
print(ai_response[:3000])

Saved to: c:\Users\egorg\Documents\RAG_минцифры\git_repo_clone\procurement_rag_system\latest_model\latest_analysis.docx

<b>1) Проверка КТРУ через сервис zakupki.gov.ru:</b>

КТРУ 22.11.11.000-00000007 найден.

Ссылка на карточку: https://zakupki.gov.ru/epz/ktru/ktruCard/commonInfo.html?itemId=22.11.11.000-00000007

Наименование совпадает с эталонной записью КТРУ.
Наименование: Шина пневматическая для легкового автомобиля
-----------------------------------------------------------------------
КТРУ 22.11.11.000-00000004 найден.

Ссылка на карточку: https://zakupki.gov.ru/epz/ktru/ktruCard/commonInfo.html?itemId=22.11.11.000-00000004

Наименование совпадает с эталонной записью КТРУ.
Наименование: Шина пневматическая для легкового автомобиля
-----------------------------------------------------------------------
КТРУ 27.20.21.000-00000021 найден.

Ссылка на карточку: https://zakupki.gov.ru/epz/ktru/ktruCard/commonInfo.html?itemId=27.20.21.000-00000021

Наименование совпадает с эталонной з

In [13]:

doc = Document()
doc.add_paragraph(ai_response)
doc.save(output_path)

print("Saved to:", output_path)
print()
print(ai_response[:3000])

Saved to: c:\Users\egorg\Documents\RAG_минцифры\git_repo_clone\procurement_rag_system\latest_model\latest_analysis.docx

<b>1) Проверка КТРУ через сервис zakupki.gov.ru:</b>

КТРУ 22.11.11.000-00000007 найден.

Ссылка на карточку: https://zakupki.gov.ru/epz/ktru/ktruCard/commonInfo.html?itemId=22.11.11.000-00000007

Наименование совпадает с эталонной записью КТРУ.
Наименование: Шина пневматическая для легкового автомобиля
-----------------------------------------------------------------------
КТРУ 22.11.11.000-00000004 найден.

Ссылка на карточку: https://zakupki.gov.ru/epz/ktru/ktruCard/commonInfo.html?itemId=22.11.11.000-00000004

Наименование совпадает с эталонной записью КТРУ.
Наименование: Шина пневматическая для легкового автомобиля
-----------------------------------------------------------------------
КТРУ 27.20.21.000-00000021 найден.

Ссылка на карточку: https://zakupki.gov.ru/epz/ktru/ktruCard/commonInfo.html?itemId=27.20.21.000-00000021

Наименование совпадает с эталонной з

# РџР РћР•РљРў РљРћРќРўР РђРљРўРђ

In [ ]:
contract_path = Path("C:\\Users\\egorg\\Documents\\RAG_РјРёРЅС†РёС„СЂС‹\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\\6_РџСЂРѕРµРєС‚_РєРѕРЅС‚СЂР°РєС‚Р°_РїРѕСЃС‚Р°РІРєР°_РјРµР±РµР»Рё_РІ2_2.docx")
plan_path = Path("C:\\Users\\egorg\\Documents\\RAG_РјРёРЅС†РёС„СЂС‹\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\\5_Р—Р°СЏРІРєР°_РЅР°_РІРєР»СЋС‡РµРЅРёРµ_РІ_РїР»Р°РЅ_РіСЂР°С„РёРє_1.docx")

parser_contract = DocumentParser(contract_path)
paragraphs_contract = parser_contract.extract_clean_text()
tables_contract = parser_contract.table_to_markdown()
contract_full_text = ("РќР°Р·РІР°РЅРёРµ: " + paragraphs_contract + "\n\n" + tables_contract).strip()

KTRY_OKPD = parser_contract.extract_table_cells_by_keyword(["РћРљРџР”", "РљРўР РЈ"])
KTRY_OKPD

import re

clean_items = []
for key in KTRY_OKPD:
    for item in KTRY_OKPD[key]:
        item = item.strip()
        item = re.sub(r"\s*;\s*", "; ", item)
        item = re.sub(r"(;\s*){2,}", "; ", item)
        item = item.rstrip("; ").strip()
        clean_items.append(item)

KTRY_OKPD_str = "\n".join(dict.fromkeys(clean_items))
print(KTRY_OKPD_str)



# РџР›РђРќ Р“Р РђР¤РРљ

In [21]:
parser_plan = PlanParser(plan_path)
plan_points = parser_plan.extract_table_kv_from_docx()
key_words = [
    "РћРљРџР”",
    "РљРўР РЈ",
    "РљРѕР»РёС‡РµСЃС‚РІРѕ"
    ]
plan_points_use = [
    plan_point
    for plan_point in plan_points
    if any(kw.lower() in plan_point.lower() for kw in key_words)
]
plan_points_str = "\n".join(plan_points_use)
print(plan_points_str)

РљРѕРґ РћРљРџР” 2 Рё РµРіРѕ РЅР°РёРјРµРЅРѕРІР°РЅРёСЏ (СЂР°СЃС€РёС„СЂРѕРІРєРё): 31.01.12.190 - РњРµР±РµР»СЊ РѕС„РёСЃРЅР°СЏ РґРµСЂРµРІСЏРЅРЅР°СЏ РїСЂРѕС‡Р°СЏ; 31.01.11.150 - РњРµР±РµР»СЊ РґР»СЏ СЃРёРґРµРЅРёСЏ, РїСЂРµРёРјСѓС‰РµСЃС‚РІРµРЅРЅРѕ СЃ РјРµС‚Р°Р»Р»РёС‡РµСЃРєРёРј РєР°СЂРєР°СЃРѕРј; 31.01.12.110 - РЎС‚РѕР»С‹ РїРёСЃСЊРјРµРЅРЅС‹Рµ РґРµСЂРµРІСЏРЅРЅС‹Рµ РґР»СЏ РѕС„РёСЃРѕРІ, Р°РґРјРёРЅРёСЃС‚СЂР°С‚РёРІРЅС‹С… РїРѕРјРµС‰РµРЅРёР№; 31.01.12.131 - РЁРєР°С„С‹ РґР»СЏ РѕРґРµР¶РґС‹ РґРµСЂРµРІСЏРЅРЅС‹Рµ; 31.01.12.139 - РЁРєР°С„С‹ РґРµСЂРµРІСЏРЅРЅС‹Рµ РїСЂРѕС‡РёРµ; 31.01.12.150 - РўСѓРјР±С‹ РѕС„РёСЃРЅС‹Рµ РґРµСЂРµРІСЏРЅРЅС‹Рµ
РљРѕРґ РїРѕР·РёС†РёРё РљРўР РЈ: 31.01.11.150-00000003 - РЎС‚СѓР» РЅР° РјРµС‚Р°Р»Р»РёС‡РµСЃРєРѕРј РєР°СЂРєР°СЃРµ; 31.01.10.000-00000004 - РЎС‚РѕР» РїРёСЃСЊРјРµРЅРЅС‹Р№; 31.01.12.131-00000001 - РЁРєР°С„ РґР»СЏ РѕРґРµР¶РґС‹ РґРµСЂРµРІСЏРЅРЅС‹Р№; 31.01.12.139-00000001 - РЁРєР°С„ РґРµСЂРµРІСЏРЅРЅС‹Р№ РґР»СЏ РґРѕРєСѓРјРµРЅС‚РѕРІ; 31.01.12.150-00000002 - РўСѓРјР±Р° РѕС„РёСЃРЅР°СЏ РґРµ

# РћРћР—

In [ ]:
OOZ_path = Path("C:\\Users\\egorg\\Documents\\RAG_РјРёРЅС†РёС„СЂС‹\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\\4. РћРћР—_РїРѕСЃС‚Р°РІРєР°_РјРµР±РµР»Рё (1).docx")

parser_ooz = DocumentParser(OOZ_path)
tables_ooz = parser_ooz.extract_table_cells_by_keyword(["РћРљРџР”", "РљРўР РЈ"])
tables_ooz

clean_items = []
for key in tables_ooz:
    for item in tables_ooz[key]:
        item = item.strip()
        item = re.sub(r"\s*;\s*", "; ", item)
        item = re.sub(r"(;\s*){2,}", "; ", item)
        item = item.rstrip("; ").strip()
        clean_items.append(item)

tables_ooz_str = "\n".join(dict.fromkeys(clean_items))
print(tables_ooz_str)


# РџРћРЇРЎРќРРўР•Р›Р¬РќРђРЇ Р—РђРџРРЎРљРђ

In [24]:
zapiska_path = Path("C:\\Users\\egorg\\Documents\\RAG_РјРёРЅС†РёС„СЂС‹\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\\7. РџРѕСЏСЃРЅРёС‚РµР»СЊРЅР°СЏ Р·Р°РїРёСЃРєР° (1).docx")
parser_zapiska = DocumentParser(zapiska_path)

paragraphs_zapiska = parser_zapiska.extract_clean_text()
tables_zapiska = parser_zapiska.table_to_markdown()
zapiska_full_text = ("РќР°Р·РІР°РЅРёРµ: " + paragraphs_zapiska + "\n\n" + tables_zapiska).strip()

print(zapiska_full_text)

РќР°Р·РІР°РЅРёРµ: РџРћРЇРЎРќРРўР•Р›Р¬РќРђРЇ Р—РђРџРРЎРљРђ

Р­Р»РµРєС‚СЂРѕРЅРЅС‹Р№ Р°СѓРєС†РёРѕРЅ РЅР° РїРѕСЃС‚Р°РІРєСѓ РѕС„РёСЃРЅРѕР№ РјРµР±РµР»Рё.

РЎРѕСЃС‚Р°РІ РїРѕСЃС‚Р°РІРєРё:

РЎС‚РѕР» РїРёСЃСЊРјРµРЅРЅС‹Р№ - 12 С€С‚.

РЎС‚РѕР» РїРёСЃСЊРјРµРЅРЅС‹Р№ - 1 С€С‚.

РќР°РґСЃС‚Р°РІРєР° РЅР°СЃС‚РѕР»СЊРЅР°СЏ - 7 С€С‚.

РўСѓРјР±Р° РѕС„РёСЃРЅР°СЏ РґРµСЂРµРІСЏРЅРЅР°СЏ - 12С€С‚.

РўСѓРјР±Р° РѕС„РёСЃРЅР°СЏ РґРµСЂРµРІСЏРЅРЅР°СЏ - 4 С€С‚.

РўСѓРјР±Р° РѕС„РёСЃРЅР°СЏ РґРµСЂРµРІСЏРЅРЅР°СЏ - 3 С€С‚.

РџРѕРґСЃС‚Р°РІРєР° РїРѕРґ СЃРёСЃС‚РµРјРЅС‹Р№ Р±Р»РѕРє - 14 С€С‚.

РљСЂРµСЃР»Рѕ РѕС„РёСЃРЅРѕРµ - 7 С€С‚.

РЁРєР°С„ РґР»СЏ РѕРґРµР¶РґС‹ РґРµСЂРµРІСЏРЅРЅС‹Р№ - 4 С€С‚.

РЁРєР°С„ РґР»СЏ РѕРґРµР¶РґС‹ РґРµСЂРµРІСЏРЅРЅС‹Р№ - 2 С€С‚.

РЁРєР°С„ РґР»СЏ РѕРґРµР¶РґС‹ РґРµСЂРµРІСЏРЅРЅС‹Р№ - 1 С€С‚.

РЁРєР°С„ РґРµСЂРµРІСЏРЅРЅС‹Р№ РґР»СЏ РґРѕРєСѓРјРµРЅС‚РѕРІ - 1 С€С‚.

РЁРєР°С„ РґРµСЂРµРІСЏРЅРЅС‹Р№ РґР»СЏ РґРѕРєСѓРјРµРЅС‚РѕРІ - 1 С€С‚.

РЁРєР°С„ РґРµСЂРµРІСЏРЅРЅС‹Р№ РґР»СЏ РґРѕРєСѓРјРµРЅС‚РѕРІ - 2 С€С‚.

РЎС‚СѓР» РЅР° 

# РћРќРњР¦Рљ РєРѕР»РёС‡РµСЃС‚РІРѕ

In [11]:
ONMCK_path = pack_dir / "2_ОЦК_метод_сопостовимых_рыночных_цен_без_учета_ПП_1875_.docx"
parser_ONMCK = DocumentParser(ONMCK_path)
table_ONMCK = parser_ONMCK.extract_rows_region(keyword = "к-т")
print(table_ONMCK)
print(parser_ONMCK.extract_clean_text())



| Ударные головки для гайковерта | к-т | 1 |
ОПРЕДЕЛЕНИЕ ЦЕНЫ КОНТРАКТА, ЗАКЛЮЧАЕМОГО С ЕДИНСТВЕННЫМ ПОСТАВЩИКОМ (ПОДРЯДЧИКОМ, ИСПОЛНИТЕЛЕМ)

В соответствии со ст. 22 Федерального закона от 05.04.2013 № 44-ФЗ "О контрактной системе в сфере закупок товаров, работ, услуг для обеспечения государственных и муниципальных нужд", методическими рекомендациями по применению методов определения начальной (максимальной) цены контракта, цены контракта, заключаемого с единственным поставщиком (подрядчиком, исполнителем) утвержденными приказом Министерства экономического развития РФ от 02.10.2013 № 567 цена контракта определена методом сопоставимых рыночных цен (анализа рынка).

Цена контракта 350 000 (триста пятьдесят тысяч) рублей 00 копеек определена по минимальному значению вышеуказанных данных.

Начальник отдела сетевого сопровождения __________________/ Якубов А.С.

13.04.2026


# РўР•РЎРўРћР’Р«Р™ Р—РђРџРЈРЎРљ РњРћР”Р•Р›Р Р”Р›РЇ РЎР РђР’РќР•РќРРЇ РљРўР РЈ РћРљРџР” Р РљРћР›РР§Р•РЎР’Рђ

In [9]:
from shared_modules.retriever import Retriever
plan_point = "РЎСЂРѕРєРё РїРѕСЃС‚Р°РІРєРё С‚РѕРІР°СЂР°, РІС‹РїРѕР»РЅРµРЅРёСЏ СЂР°Р±РѕС‚," \
" РѕРєР°Р·Р°РЅРёСЏ СѓСЃР»СѓРі РїРѕ РєРѕРЅС‚СЂР°РєС‚Сѓ: РІ С‚РµС‡РµРЅРёРµ 45 (СЃРѕСЂРѕРєР° РїСЏС‚Рё)" \
" РєР°Р»РµРЅРґР°СЂРЅС‹С… РґРЅРµР№ СЃ РґР°С‚С‹ Р·Р°РєР»СЋС‡РµРЅРёСЏ РєРѕРЅС‚СЂР°РєС‚Р°"

parser_onmck = DocumentParser(ONMCK_path)
parser_ONMCK = DocumentParser(ONMCK_path)
parser_zapiska = DocumentParser(zapiska_path)
parser_ooz = DocumentParser(OOZ_path)

paragraphs_zapiska = parser_zapiska.extract_clean_text()
tables_zapiska = parser_zapiska.table_to_markdown()

zapiska_full_text = ("РќР°Р·РІР°РЅРёРµ: " + paragraphs_zapiska + "\n\n" + tables_zapiska).strip()
paragraphs_contract = parser_contract.extract_clean_text()
contract_full_text = paragraphs_contract.strip()
ooz_plain_text = parser_ooz.extract_clean_text()
onmck_plain_text = parser_onmck.extract_clean_text()

faiss = Retriever(embeddings=get_embeddings())
retriever = faiss.create_retriever(texts=[contract_full_text, zapiska_full_text, ooz_plain_text, onmck_plain_text], n=20)
docs = retriever.invoke(plan_point)
print("=" * 80)
print(f"plan_point {1}: {plan_point}")
print("-" * 80)

for j, doc in enumerate(docs, start=1):
    print(f"chunk {j}:")
    print(doc.page_content[:1500])
    print("-" * 80)

plan_point 1: РЎСЂРѕРєРё РїРѕСЃС‚Р°РІРєРё С‚РѕРІР°СЂР°, РІС‹РїРѕР»РЅРµРЅРёСЏ СЂР°Р±РѕС‚, РѕРєР°Р·Р°РЅРёСЏ СѓСЃР»СѓРі РїРѕ РєРѕРЅС‚СЂР°РєС‚Сѓ: РІ С‚РµС‡РµРЅРёРµ 45 (СЃРѕСЂРѕРєР° РїСЏС‚Рё) РєР°Р»РµРЅРґР°СЂРЅС‹С… РґРЅРµР№ СЃ РґР°С‚С‹ Р·Р°РєР»СЋС‡РµРЅРёСЏ РєРѕРЅС‚СЂР°РєС‚Р°
--------------------------------------------------------------------------------
chunk 1:
СЃРёСЃС‚РµРјС‹ РІ СЃС„РµСЂРµ Р·Р°РєСѓРїРѕРє (РґР°Р»РµРµ - РјРµСЃС‚Рѕ РґРѕСЃС‚Р°РІРєРё), РІ СЃСЂРѕРє РІ С‚РµС‡РµРЅРёРµ 45 (СЃРѕСЂРѕРєР° РїСЏС‚Рё) РєР°Р»РµРЅРґР°СЂРЅС‹С… РґРЅРµР№ СЃ РґР°С‚С‹ Р·Р°РєР»СЋС‡РµРЅРёСЏ РљРѕРЅС‚СЂР°РєС‚Р°. Р”Р°С‚Р° Р·Р°РєР»СЋС‡РµРЅРёСЏ РљРѕРЅС‚СЂР°РєС‚Р° РЅРµ РІС…РѕРґРёС‚ РІ СЃСЂРѕРє РїРѕСЃС‚Р°РІРєРё РўРѕРІР°СЂР°
--------------------------------------------------------------------------------
chunk 2:
РўР°Р±Р»РёС†Р° 1 РњРµСЃС‚Рѕ РїРѕСЃС‚Р°РІРєРё РўРѕРІР°СЂРѕРІ

РЎСЂРѕРє РїРѕСЃС‚Р°РІРєРё: РІ С‚РµС‡РµРЅРёРµ 45 РєР°Р»РµРЅРґР°СЂРЅС‹С… РґРЅРµР№ СЃ РґР°С‚С‹ Р·Р°РєР»СЋС‡РµРЅРёСЏ РєРѕРЅС‚СЂР°РєС‚Р°.

РҐР°СЂР°РєС‚РµС

In [14]:
from shared_modules.retriever import BM25TextRetriever

bm25 = BM25TextRetriever()
retriever = bm25.create_retriever(
    texts=[contract_full_text, zapiska_full_text, ooz_plain_text, onmck_plain_text],
    n=7,
)

docs = retriever.invoke(plan_point)

print("=" * 80)
print(f"plan_point 1: {plan_point}")
print("-" * 80)

for j, doc in enumerate(docs, start=1):
    print(f"chunk {j}:")
    print(doc.page_content[:1500])
    print("-" * 80)


plan_point 1: РЎСЂРѕРєРё РїРѕСЃС‚Р°РІРєРё С‚РѕРІР°СЂР°, РІС‹РїРѕР»РЅРµРЅРёСЏ СЂР°Р±РѕС‚, РѕРєР°Р·Р°РЅРёСЏ СѓСЃР»СѓРі РїРѕ РєРѕРЅС‚СЂР°РєС‚Сѓ: РІ С‚РµС‡РµРЅРёРµ 45 (СЃРѕСЂРѕРєР° РїСЏС‚Рё) РєР°Р»РµРЅРґР°СЂРЅС‹С… РґРЅРµР№ СЃ РґР°С‚С‹ Р·Р°РєР»СЋС‡РµРЅРёСЏ РєРѕРЅС‚СЂР°РєС‚Р°
--------------------------------------------------------------------------------
chunk 1:
СЃРёСЃС‚РµРјС‹ РІ СЃС„РµСЂРµ Р·Р°РєСѓРїРѕРє (РґР°Р»РµРµ - РјРµСЃС‚Рѕ РґРѕСЃС‚Р°РІРєРё), РІ СЃСЂРѕРє РІ С‚РµС‡РµРЅРёРµ 45 (СЃРѕСЂРѕРєР° РїСЏС‚Рё) РєР°Р»РµРЅРґР°СЂРЅС‹С… РґРЅРµР№ СЃ РґР°С‚С‹ Р·Р°РєР»СЋС‡РµРЅРёСЏ РљРѕРЅС‚СЂР°РєС‚Р°. Р”Р°С‚Р° Р·Р°РєР»СЋС‡РµРЅРёСЏ РљРѕРЅС‚СЂР°РєС‚Р° РЅРµ РІС…РѕРґРёС‚ РІ СЃСЂРѕРє РїРѕСЃС‚Р°РІРєРё РўРѕРІР°СЂР°
--------------------------------------------------------------------------------
chunk 2:
РўР°Р±Р»РёС†Р° в„–1 РњРµСЃС‚Рѕ РїРѕСЃС‚Р°РІРєРё РўРѕРІР°СЂРѕРІ

РЎСЂРѕРє РїРѕСЃС‚Р°РІРєРё: РІ С‚РµС‡РµРЅРёРµ 45 (СЃРѕСЂРѕРєР° РїСЏС‚Рё) РєР°Р»РµРЅРґР°СЂРЅС‹С… РґРЅРµР№ СЃ РґР°С‚С‹ Р·Р°РєР»СЋС‡РµРЅРёСЏ РєРѕРЅС‚С

In [ ]:
# contract_path = Path(r"C:\Users\egorg\Documents\RAG_РјРёРЅС†РёС„СЂС‹\git_repo_clone\Documents-verification-Mintsifry\doci_primery\Р”Р›РЇ Р“РџРў\4_РљРѕРЅС‚СЂР°РєС‚_РЅР°_РїРѕСЃС‚Р°РІРєСѓ_РјРѕРЅРѕР±Р»РѕРєРѕРІ_1.docx")
# plan_path = Path(r"C:\Users\egorg\Documents\RAG_РјРёРЅС†РёС„СЂС‹\git_repo_clone\Documents-verification-Mintsifry\doci_primery\Р”Р›РЇ Р“РџРў\1. Р—Р°СЏРІРєР° РІ РџР“ (1).docx")
# zapiska_path = Path(r"C:\Users\egorg\Documents\RAG_РјРёРЅС†РёС„СЂС‹\git_repo_clone\Documents-verification-Mintsifry\doci_primery\Р”Р›РЇ Р“РџРў\5. РџРѕСЏСЃРЅРёС‚РµР»СЊРЅР°СЏ Р·Р°РїРёСЃРєР° (1).docx")
# OOZ_path = Path(r"C:\Users\egorg\Documents\RAG_РјРёРЅС†РёС„СЂС‹\git_repo_clone\Documents-verification-Mintsifry\doci_primery\Р”Р›РЇ Р“РџРў\3.РћРћР— РЅР° РїРѕСЃС‚Р°РІРєСѓ РјРѕРЅРѕР±Р»РѕРєРѕРІ (1).docx")
# ONMCK_path = Path(r"C:\Users\egorg\Documents\RAG_РјРёРЅС†РёС„СЂС‹\git_repo_clone\Documents-verification-Mintsifry\doci_primery\Р”Р›РЇ Р“РџРў\2. РћРќРњР¦Рљ (1).docx")
# Obrasheniye_path = Path(r"C:\Users\egorg\Documents\RAG_РјРёРЅС†РёС„СЂС‹\git_repo_clone\Documents-verification-Mintsifry\doci_primery\Р”Р›РЇ Р“РџРў\0_РћР±СЂР°С‰РµРЅРёРµ_Рѕ_РїСЂРѕРІРµРґРµРЅРёРё_Р·Р°РєСѓРїРєРё_1.docx")


In [16]:
output_path = project_root / "latest_model" / "latest_analysis.docx"
doc.save(output_path)

print("Saved to:", output_path)
print()
print(ai_response[:3000])

Saved to: c:\Users\egorg\Documents\RAG_РјРёРЅС†РёС„СЂС‹\git_repo_clone\Documents-verification-Mintsifry\latest_model\latest_analysis.docx


----------------------------------------------------------------------------------

----------------------------------------------------------------------------------
РљРѕРґ 26.20.15.140 РЅР°РїСЂСЏРјСѓСЋ РЅРµ РЅР°Р№РґРµРЅ. РќР°Р№РґРµРЅ СЂРѕРґРёС‚РµР»СЊСЃРєРёР№ РєРѕРґ 26.20.15 РІ С‚Р°Р±Р»РёС†Рµ 'РџР РР›РћР–Р•РќРР• N 2 Рє РїРѕСЃС‚Р°РЅРѕРІР»Рµ...'. Р­С‚Р°Р»РѕРЅРЅРѕРµ РЅР°РёРјРµРЅРѕРІР°РЅРёРµ: РњР°С€РёРЅС‹ РІС‹С‡РёСЃР»РёС‚РµР»СЊРЅС‹Рµ СЌР»РµРєС‚СЂРѕРЅРЅС‹Рµ С†РёС„СЂРѕРІС‹Рµ РїСЂРѕС‡РёРµ, СЃРѕРґРµСЂР¶Р°С‰РёРµ РёР»Рё РЅРµ СЃРѕРґРµСЂР¶Р°С‰РёРµ РІ РѕРґРЅРѕРј РєРѕСЂРїСѓСЃРµ РѕРґРЅРѕ РёР»Рё РґРІР° РёР· СЃР»РµРґСѓСЋС‰РёС… СѓСЃС‚СЂРѕР№СЃС‚РІ РґР»СЏ Р°РІС‚РѕРјР°С‚РёС‡РµСЃРєРѕР№ РѕР±СЂР°Р±РѕС‚РєРё РґР°РЅРЅС‹С…: Р·Р°РїРѕРјРёРЅР°СЋС‰РёРµ СѓСЃС‚СЂРѕР№СЃС‚РІР°, СѓСЃС‚СЂРѕР№СЃС‚РІР° РІРІРѕРґР°, СѓСЃС‚СЂРѕР№СЃС‚РІР° РІС‹РІРѕРґР°. РџСЂРѕРІРµСЂСЊС‚Рµ СЃРѕРѕС‚РІРµС‚СЃС

In [ ]:
# import sys
# from pathlib import Path
# from docx import Document

# project_root = Path(r"C:\Users\egorg\Documents\RAG_РјРёРЅС†РёС„СЂС‹\git_repo_clone\Documents-verification-Mintsifry")
# if str(project_root) not in sys.path:
#     sys.path.insert(0, str(project_root))

# from latest_model.ai_service import get_ai_service

# contract_path = Path(r"C:\Users\egorg\Documents\RAG_РјРёРЅС†РёС„СЂС‹\git_repo_clone\Documents-verification-Mintsifry\doci_primery\6_РџСЂРѕРµРєС‚_РєРѕРЅС‚СЂР°РєС‚Р°_РїРѕСЃС‚Р°РІРєР°_РјРµР±РµР»Рё_РІ2_2.docx")
# plan_path = Path(r"C:\Users\egorg\Documents\RAG_РјРёРЅС†РёС„СЂС‹\git_repo_clone\Documents-verification-Mintsifry\doci_primery\5_Р—Р°СЏРІРєР°_РЅР°_РІРєР»СЋС‡РµРЅРёРµ_РІ_РїР»Р°РЅ_РіСЂР°С„РёРє_1.docx")

# ai_service = get_ai_service()

# result = ai_service.process_query(Р¦
#     doc1_path=str(plan_path),
#     doc2_path=str(contract_path),
# )

# ai_response = result["ai_response"]

# output_path = project_root / "latest_model" / "latest_analysis.docx"

# doc = Document()
# doc.add_paragraph(ai_response)
# doc.save(output_path)

# print("Saved to:", output_path)
# print()
# print(ai_response[:3000])
